# Feature selection and preprocessing

## Import dask and set up client for resource dashboard

In [ ]:
from dask import dataframe as dd
from dask.distributed import Client # only for dashboard

client = Client() # only for dashboard
print(client.dashboard_link) # only for dashboard

## Read data set and select features we are interested in

In [ ]:
df = dd.read_json("arxiv-metadata-oai-snapshot-v282.json", blocksize="16MB", dtype={"id": "string"}) # ID gets read as a float otherwise
print(df.dtypes)

df["update_date"] = dd.to_datetime(df["update_date"], format="%Y-%m-%d")

# Print data types right after reading JSON dataset
# print(df.dtypes)

df = df[["id", "update_date", "title", "abstract", "categories"]]
# Print data types after selecting columns
print(df.dtypes)

id                string
submitter         string
authors           string
title             string
comments          string
journal-ref       string
doi               string
report-no         string
categories        string
license           string
abstract          string
versions          string
update_date       string
authors_parsed    string
dtype: object
id                     string
update_date    datetime64[us]
title                  string
abstract               string
categories             string
dtype: object


## Keep only papers younger than ~5 years (2019-01-01 onwards)

In [3]:
df = df[df["update_date"] >= "2019-01-01"]

## Deal with empty and very short values in the 'title' and 'abstract' columns

In [ ]:
show_rows_to_be_removed = True
if show_rows_to_be_removed:
    # Find rows with any null values
    empty_rows = df[df.isnull().any(axis=1)]
    print(f"There are {len(empty_rows)} rows with null values.")

    # Find rows with empty strings in the 'title' column
    empty_title_rows = df[df['title'] == '']
    print(f"There are {len(empty_title_rows)} rows with empty string as title.")

    # Find rows with empty strings in the 'abstract' column
    empty_abstract_rows = df[df['abstract'] == '']
    print(f"There are {len(empty_abstract_rows)} rows with empty string as abstract.")

    # Find rows with less than 10 characters in the 'title' column
    short_title_rows = df[df['title'].str.len() < 10]
    short_title_rows_len = len(short_title_rows)
    print(f"There are {short_title_rows_len} rows with short titles (less than 10 characters).")

    # Find rows with less than 10 characters in the 'abstract' column
    short_abstract_rows = df[df['abstract'].str.len() < 10]
    short_abstract_rows_len = len(short_abstract_rows)
    print(f"There are {short_abstract_rows_len} rows with short abstracts (less than 10 characters).")

In [ ]:
len_before = len(df)
df = df[df['title'].str.len() >= 10]
len_titles_removed = len(df)
print(f"    {len_before - len_titles_removed} rows with short titles dropped")

df = df[df['abstract'].str.len() >= 10]
print(f"    {len_titles_removed - len(df)} rows with short abstracts dropped")

    132 rows with short titles dropped
    3 rows with short abstracts dropped


## Clean titles and abstracts

In [6]:
# Define regex patterns for cleaning
RE_LATEX_INLINE = r"\$.*?\$"
RE_LATEX_COMMAND = r"\\[a-zA-Z]+"
RE_WHITESPACE = r"\s+"

# Clean the 'title' and 'abstract' columns
for column in ("title", "abstract"):
    print(f"Cleaning column '{column}'...")
    df[column] = (
        df[column]
            .str.lower()
            .str.replace(RE_LATEX_INLINE, "", regex=True) # inline math
            .str.replace(RE_LATEX_COMMAND, "", regex=True)  # LaTeX commands
            .str.replace(RE_WHITESPACE, " ", regex=True) # replace multiple whitespace with single space
            .str.strip()
    )
    print(f"Done cleaning '{column}' column.")

Cleaning column 'title'...
Done cleaning 'title' column.
Cleaning column 'abstract'...
Done cleaning 'abstract' column.


## Drop duplicate titles and abstracts

In [7]:
original_len = len(df)

df = df.drop_duplicates(subset=['title']) # 1636 duplicates
len_title_dedup = len(df)
print(f"Dropped {original_len - len_title_dedup} rows with duplicate titles.")

df = df.drop_duplicates(subset=['abstract']) # 200 duplicates
len_abstract_dedup = len(df)
print(f"Dropped {len_title_dedup - len_abstract_dedup} rows with duplicate abstracts.")

print(f"Dropped total of {original_len - len_abstract_dedup} rows with duplicate titles and abstracts.")

Dropped 1636 rows with duplicate titles.
Dropped 200 rows with duplicate abstracts.
Dropped total of 1836 rows with duplicate titles and abstracts.


## Write preprocessed dataset for BERTopic to parquet

In [8]:
df.to_parquet("artifacts/Preprocessing/arxiv-metadata-cleaned_BERTopic.parquet")

# Further preprocessing for LDA
LDA benefits from a few more preprocessing steps, such as stopword removal, that actually weaken BERTopic performance so we run these at the end and write a separate preprocessing artifact for each.
BERTopic leverages transformers to generate embeddings which benefit being given more 'raw' language data as they can extract semantics from words that will be removed as stopwords going forward.

In [9]:
import nltk
import re

from custom_stopwords import custom_stopwords

nltk.download('stopwords')
stop_words = set(nltk.corpus.stopwords.words('english'))
stop_words = stop_words.union(custom_stopwords)

nltk.download('wordnet')
lemmatizer = nltk.stem.WordNetLemmatizer()

def further_preprocessing(text):
    # remove any char that is not a-z, 0-9, -, or whitespace
    # (columns are already lowered at this point)
    text = re.sub(r'[^a-z0-9-\s]+', '', text)

    # tokenize
    tokens = text.split()

    # remove very short and very long tokens
    tokens = [t for t in tokens if 3 <= len(t) <= 20]

    # remove stopwords
    tokens = [t for t in tokens if t not in stop_words]

    # lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return tokens


for column in ("title", "abstract"):
    df[column] = df[column].apply(further_preprocessing, meta=(column, 'string'))

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/azureuser/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/azureuser/nltk_data...


## Write (further) preprocessed dataset for LDA to parquet

In [10]:
import pyarrow

schema = pyarrow.schema([
    ("id", pyarrow.string()),
    ("update_date", pyarrow.timestamp("us")),
    ("title", pyarrow.list_(pyarrow.string())),
    ("abstract", pyarrow.list_(pyarrow.string())),
    ("categories", pyarrow.string()),
])

df.to_parquet("artifacts/Preprocessing/arxiv-metadata-cleaned_LDA.parquet", schema=schema)

# Run optionally to shutdown Dask client

In [11]:
client.close()